# Laboratorio de clasificación 3

|                |   |
:----------------|---|
| **Nombre**     |  Claudio Alejo Encarnación Martínez |
| **Fecha**      |  12 octubre 2026 |
| **Expediente** |  750597 | 

Una regresión logística simple toma **P** predictores y usa una función no lineal para convertir la respuesta binaria **Y** en probabilidades entre $0$ y $1$.

La visualización de este escenario es puntos separados por una línea recta.

<center><img src="https://wirelesspi.com/wp-content/uploads/2022/08/figure-logistic-regression-decision-boundary-2-570x550.png"></center>

Para casos donde los datos no son linealmente separables usamos el truco del kernel para mapear los datos a un espacio de Hilbert.

Imagina un escenario donde tienes dos predictores $x_1, x_2$ y la salida de estos predictores se comporta de esta manera:

| **$x_1$** | **$x_2$** | Salida |
:-----------|-----------|--------|
| 0         | 0         | 0      |
| 0         | 1         | 1      |
| 1         | 0         | 1      |
| 1         | 1         | 0      |


Imagínalo como el interruptos de luz de un foco en una escalera. Se puede prender y apagar desde arriba y desde abajo.

<center><img src="https://www.researchgate.net/profile/Brian-Spears/publication/322048911/figure/fig4/AS:667820685807618@1536232267250/The-exclusive-or-XOR-function-is-a-nonlinear-function-that-returns-0-when-its-two.png"></center>

¿Cómo podemos separar estos datos usando sólo regresión logística? Sin importar cómo usemos una sola línea, nunca se van a poder separar todos los casos, por lo que tendríamos situaciones donde siempre habría error. 

Solución: ¿Por qué nos estamos limitando a una sola línea?

<center><img src="https://pabloinsente.github.io/assets/post-5/XOR-linear.png"></center>

La regresión logística sola es un método de aprendizaje estadístico convexo. Combinar dos regresiones logísticas para clasificar datos no linealmente separables convierte el problema a uno no convexo. ¿Por qué?

- Una regresión logística simple es convexa porque el modelo es lineal en los parámetros y la función de pérdida (log-loss) es convexa. Al combinar dos regresiones logísticas, por ejemplo en una capa oculta con activaciones sigmoideas y una capa de salida también sigmoidea, la salida final depende de productos y composiciones de funciones no lineales. Esa composición hace que la pérdida resultante ya no sea convexa en el conjunto de todos los pesos; aparecen múltiples mínimos locales, mesetas y puntos silla.

¿Qué significa que una función sea convexa?

- Una función convexa es aquella en la que, para cualesquiera dos puntos en su dominio, el segmento de línea que los une se encuentra por encima o sobre la gráfica de la función. Matemáticamente, una función f es convexa si para todos (x, y) en su dominio y para todo \(\lambda \in [0,1]\) se cumple:


¿Por qué nos importa que una función sea convexa o no? ¿Qué impacto tiene para los resultados?

- En optimización convexa, cualquier mínimo local es mínimo global. Esto garantiza que métodos como descenso de gradiente converjan al óptimo global (bajo condiciones adecuadas), sin depender fuertemente de la inicialización. En problemas no convexos, como las redes neuronales, pueden existir múltiples mínimos locales y puntos silla; el resultado depende de la inicialización, la tasa de aprendizaje, el optimizador, etc. Por eso se usan múltiples reinicios, validación, early stopping y técnicas estocásticas para encontrar buenas soluciones, aunque no se garantice el óptimo global.

## Ejercicio 1

Utilizaremos el clasificador `MLPClassifier` de la librería `sklearn`.

Entra a la documentación y describe cómo funciona este objeto (qué entradas necesita, qué salidas da, descripción de sus hiperparámetros).

- MLPClassifier de sklearn implementa un perceptrón multicapa para clasificación.

Entradas:
- X: array-like de forma (n_samples, n_features).
- y: array-like de forma (n_samples,) con etiquetas de clase.

Salidas:
- `predict(X)`: devuelve las etiquetas predichas.
- `predict_proba(X)`: devuelve probabilidades por clase.
- `predict_log_proba(X)`: log-probabilidades.

Atributos tras el ajuste: classes_, loss_, coefs_, intercepts_, n_iter_, out_activation_.

Hiperparámetros principales:
- `hidden_layer_sizes`: tupla con el número de neuronas por capa oculta. Default (100,).
- `activation`: función de activación: 'identity', 'logistic', 'tanh', 'relu'. Default 'relu'.
- `solver`: algoritmo de optimización: 'lbfgs', 'sgd', 'adam'. Default 'adam'.
- `alpha`: regularización L2. Default 0.0001.
- `batch_size`: tamaño de lote para 'sgd' y 'adam'. Default 'auto'.
- `learning_rate`: esquema de tasa de aprendizaje para 'sgd': 'constant', 'invscaling', 'adaptive'.
- `learning_rate_init`: tasa inicial. Default 0.001.
- `max_iter`: número máximo de iteraciones. Default 200.
- `shuffle`: mezclar muestras en cada iteración. Default True.
- `random_state`: semilla para reproducibilidad.
- `tol`: tolerancia para criterio de parada.
- `early_stopping`: si es True, usa validación interna para detener el entrenamiento.
- `validation_fraction`: proporción de datos para validación si early_stopping=True.
- `n_iter_no_change`: paciencia para early stopping.
- `beta_1`, `beta_2`, `epsilon`: parámetros de Adam.
- `momentum`, `nesterovs_momentum`: para SGD.

## Ejercicio 2

Utiliza el objeto de perceptrón multicapa para clasificar si un tumor es maligno o benigno con el dataset `Brain Tumor Dataset`. Utiliza un cross-validation de train-test de 70/30. Sigue todos los pasos de EDA y limpieza vistos.

### Importación de librerías

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix



### EDA

In [3]:
df = pd.read_csv("/Users/claudioalejoencarnacionmartinez/Documents/SEMESTRE OTOÑO 2026/LAB APRENDIZAJE ESTADÍSTICO/LABORATORIO-DE-APRENDIZAJE-ESTAD-STICO/SEGUNDO PARCIAL/brain_tumor_dataset.csv")
df.head()

,Patient_ID,Age,Gender,Tumor_Type,Tumor_Size,Location,Histology,Stage,Symptom_1,Symptom_2,Symptom_3,Radiation_Treatment,Surgery_Performed,Chemotherapy,Survival_Rate,Tumor_Growth_Rate,Family_History,MRI_Result,Follow_Up_Required
0,1,73,Male,Malignant,5.375612,Temporal,Astrocytoma,III,Vision Issues,Seizures,Seizures,No,No,No,51.312579,0.111876,No,Positive,Yes
1,2,26,Male,Benign,4.847098,Parietal,Glioblastoma,II,Headache,Headache,Nausea,Yes,Yes,Yes,46.373273,2.165736,Yes,Positive,Yes
2,3,31,Male,Benign,5.588391,Parietal,Meningioma,I,Vision Issues,Headache,Seizures,No,No,No,47.072221,1.884228,No,Negative,No
3,4,29,Male,Malignant,1.436600,Temporal,Medulloblastoma,IV,Vision Issues,Seizures,Headache,Yes,No,Yes,51.853634,1.283342,Yes,Negative,No
4,5,54,Female,Benign,2.417506,Parietal,Glioblastoma,I,Headache,Headache,Seizures,No,No,Yes,54.708987,2.069477,No,Positive,Yes


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Patient_ID           20000 non-null  int64  
 1   Age                  20000 non-null  int64  
 2   Gender               20000 non-null  str    
 3   Tumor_Type           20000 non-null  str    
 4   Tumor_Size           20000 non-null  float64
 5   Location             20000 non-null  str    
 6   Histology            20000 non-null  str    
 7   Stage                20000 non-null  str    
 8   Symptom_1            20000 non-null  str    
 9   Symptom_2            20000 non-null  str    
 10  Symptom_3            20000 non-null  str    
 11  Radiation_Treatment  20000 non-null  str    
 12  Surgery_Performed    20000 non-null  str    
 13  Chemotherapy         20000 non-null  str    
 14  Survival_Rate        20000 non-null  float64
 15  Tumor_Growth_Rate    20000 non-null  float64
 1

In [5]:
df.describe()

,Patient_ID,Age,Tumor_Size,Survival_Rate,Tumor_Growth_Rate
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,10000.500000,49.433700,5.236552,70.131699,1.547701
std,5773.647028,17.389296,2.754359,17.270834,0.835995
min,1.000000,20.000000,0.500410,40.002386,0.100017
25%,5000.750000,34.000000,2.836686,55.224439,0.830433
50%,10000.500000,49.000000,5.200675,70.305066,1.541821
75%,15000.250000,65.000000,7.624299,85.043395,2.262484
max,20000.000000,79.000000,9.999420,99.998125,2.999835


### Limpieza 

In [6]:
df = df.drop_duplicates()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Patient_ID           20000 non-null  int64  
 1   Age                  20000 non-null  int64  
 2   Gender               20000 non-null  str    
 3   Tumor_Type           20000 non-null  str    
 4   Tumor_Size           20000 non-null  float64
 5   Location             20000 non-null  str    
 6   Histology            20000 non-null  str    
 7   Stage                20000 non-null  str    
 8   Symptom_1            20000 non-null  str    
 9   Symptom_2            20000 non-null  str    
 10  Symptom_3            20000 non-null  str    
 11  Radiation_Treatment  20000 non-null  str    
 12  Surgery_Performed    20000 non-null  str    
 13  Chemotherapy         20000 non-null  str    
 14  Survival_Rate        20000 non-null  float64
 15  Tumor_Growth_Rate    20000 non-null  float64
 1

### Definición de variables

In [7]:
y = df["Tumor_Type"].map({"Benign": 0, "Malignant": 1})
X = df.drop(["Tumor_Type","Patient_ID"], axis=1)

In [8]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
print("\nNuméricas:", num_cols)
print("Categóricas:", cat_cols)


Numéricas: ['Age', 'Tumor_Size', 'Survival_Rate', 'Tumor_Growth_Rate']
Categóricas: ['Gender', 'Location', 'Histology', 'Stage', 'Symptom_1', 'Symptom_2', 'Symptom_3', 'Radiation_Treatment', 'Surgery_Performed', 'Chemotherapy', 'Family_History', 'MRI_Result', 'Follow_Up_Required']


/var/folders/yw/qyjd88g55q12jrgy7__880_w0000gn/T/ipykernel_15014/848284252.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns.tolist()


### Pipelines

In [9]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols)
    ]
)

### Construcción del modelo

In [10]:
mlp = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    max_iter=500,
    random_state=42
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("mlp", mlp)
])

### Train-test split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.30,random_state=42)

### Entrenamiento y evaluación

In [12]:
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, y_pred))
print("\nMatriz de confusión:")
print(confusion_matrix(y_test, y_pred))

print("\nReporte de clasificación:")
print(classification_report(
    y_test, y_pred,
    target_names=["Benign", "Malignant"]
))

/opt/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:792: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")



Accuracy: 0.5105

Matriz de confusión:
[[1445 1517]
 [1420 1618]]

Reporte de clasificación:
              precision    recall  f1-score   support

      Benign       0.50      0.49      0.50      2962
   Malignant       0.52      0.53      0.52      3038

    accuracy                           0.51      6000
   macro avg       0.51      0.51      0.51      6000
weighted avg       0.51      0.51      0.51      6000



¿Qué métrico crees que sea mejor para este ejercicio? Escoge los 3 métricos que crees más apropiados para esta predicción, argumenta en pro y en contra del top 3. Elige uno para optimizar (justifica tu respuesta).

1. Pues como ya se ha discutido anteriormente, en casos clínicos, el error más grave son falsos negativos. Por lo que me iría más a medir el Recall. Además es la métrica estándar en oncología para evaluar modelos de clasificación. La desventaja es que ignora los falsos positivos, por lo que se puede caer en la trampa de predecir todo como maligno para tener un recall perfecto.
2. El F1-score ya toma en cuenta los falsos positivos que el Recall ignora. Esto lo hace una métrica más robusta cuando se quiere considerar tanto los falsos negativos como los falsos positivos. Sin embargo, esto mismo es su desventaja porque en verdad puede no reflejar adecuadamente la gravedad de los falsos negativos en contextos clínicos, pero por ello se usará después de analizar el recall.
3. ROC-AUC, área bajo la curva TPR vs FPR. Esta métrica es útil para evaluar la capacidad del modelo para discriminar entre clases, independientemente del umbral de decisión, sin embargo por eso mismo, no refleja directamente la precisión o el recall en un umbral específico (o sea no me ayuda a evaluar el desempeño del modelo en un umbral particular).


## Ejercicio 3

Modifica los hiperparámetros para elevar el métrico elegido. Intenta con 10 sets de hiperparámetros distintos. Crea una tabla que muestre los resultados.

In [13]:
from sklearn.metrics import (recall_score, precision_score,f1_score, roc_auc_score, accuracy_score)
   

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc",  StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh",  OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y)
param_sets = [
    ("S1  baseline (32,16) relu adam",  {"hidden_layer_sizes": (32, 16), "alpha": 1e-4, "learning_rate_init": 1e-3, "activation": "relu", "solver": "adam"}),
    ("S2  más capacidad (64,32)",       {"hidden_layer_sizes": (64, 32), "alpha": 1e-4, "learning_rate_init": 1e-3, "activation": "relu", "solver": "adam"}),
    ("S3  más capacidad (128,64)",      {"hidden_layer_sizes": (128, 64),"alpha": 1e-4, "learning_rate_init": 1e-3, "activation": "relu", "solver": "adam"}),
    ("S4  una capa (64,)",              {"hidden_layer_sizes": (64,),    "alpha": 1e-4, "learning_rate_init": 1e-3, "activation": "relu", "solver": "adam"}),
    ("S5  mucha regularización α=1e-2", {"hidden_layer_sizes": (32, 16), "alpha": 1e-2, "learning_rate_init": 1e-3, "activation": "relu", "solver": "adam"}),
    ("S6  learning rate alto 1e-2",     {"hidden_layer_sizes": (32, 16), "alpha": 1e-4, "learning_rate_init": 1e-2, "activation": "relu", "solver": "adam"}),
    ("S7  tanh + lr alto",              {"hidden_layer_sizes": (64, 32), "alpha": 1e-3, "learning_rate_init": 1e-2, "activation": "tanh", "solver": "adam"}),
    ("S8  tres capas (128,64,32)",      {"hidden_layer_sizes": (128, 64, 32), "alpha": 1e-4, "learning_rate_init": 1e-3, "activation": "relu", "solver": "adam"}),
    ("S9  solver SGD momentum 0.9",     {"hidden_layer_sizes": (64, 32), "alpha": 1e-3, "learning_rate_init": 1e-2, "activation": "relu", "solver": "sgd", "momentum": 0.9}),
    ("S10 solver lbfgs (red pequeña)",  {"hidden_layer_sizes": (32, 16), "alpha": 1e-4, "activation": "relu", "solver": "lbfgs"}),
]

results = []
for name, params in param_sets:
    mlp = MLPClassifier(max_iter=1000, random_state=42, **params)
    model = Pipeline([("pre", preprocessor), ("mlp", mlp)])
    model.fit(X_train, y_train)

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "Set":                  name,
        "Recall (Malignant)":   recall_score(y_test, y_pred),
        "Precision (Malignant)":precision_score(y_test, y_pred),
        "F1 (Malignant)":       f1_score(y_test, y_pred),
        "ROC-AUC":              roc_auc_score(y_test, y_proba),
        "Accuracy":             accuracy_score(y_test, y_pred),
    })

tabla = (pd.DataFrame(results)
           .sort_values("Recall (Malignant)", ascending=False)
           .reset_index(drop=True))

pd.set_option("display.float_format", "{:.4f}".format)
print(tabla.to_string(index=False))

                            Set  Recall (Malignant)  Precision (Malignant)  F1 (Malignant)  ROC-AUC  Accuracy
    S6  learning rate alto 1e-2              0.5264                 0.4967          0.5111   0.4980    0.4950
     S3  más capacidad (128,64)              0.5198                 0.5006          0.5100   0.5068    0.4992
S5  mucha regularización α=1e-2              0.5108                 0.4955          0.5030   0.4922    0.4938
 S10 solver lbfgs (red pequeña)              0.5101                 0.5005          0.5053   0.4968    0.4990
             S7  tanh + lr alto              0.5008                 0.5086          0.5047   0.5085    0.5070
     S8  tres capas (128,64,32)              0.4958                 0.4995          0.4977   0.4940    0.4980
             S4  una capa (64,)              0.4932                 0.4987          0.4959   0.4981    0.4972
    S9  solver SGD momentum 0.9              0.4919                 0.5082          0.4999   0.5095    0.5065
 S1  basel

/opt/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:606: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


In [14]:
# Comprobación de alineación real
assert (X.index == y.index).all()
# O más fuerte: mira una fila concreta y su etiqueta
print(df.loc[0, ["Tumor_Type"]].values, y.iloc[0])
print(df.loc[1, ["Tumor_Type"]].values, y.iloc[1])

<StringArray>
['Malignant']
Length: 1, dtype: str 1
<StringArray>
['Benign']
Length: 1, dtype: str 0


In [16]:
Xt = preprocessor.fit_transform(X_train)
print(pd.Series(Xt.std(axis=0)).describe())
# Si min std < 0.01, hay columnas "muertas" para el MLP

count   42.0000
mean     0.5093
std      0.1641
min      0.4303
25%      0.4328
50%      0.4352
75%      0.5000
max      1.0000
dtype: float64


In [18]:
Xt = preprocessor.fit_transform(X_train)
print("Shape post-preprocesador:", Xt.shape)
print("Columnas constantes:", (Xt.std(axis=0) == 0).sum())

Shape post-preprocesador: (14000, 42)
Columnas constantes: 0


In [19]:
print("Iteraciones usadas:", model.named_steps["mlp"].n_iter_)
print("Loss final train:", model.named_steps["mlp"].loss_)
# Si n_iter_ == 2000 → no convergió
# Si loss_ > 0.65 → sigue cerca del azar

Iteraciones usadas: 1000
Loss final train: 0.5686593174195217


RF ROC-AUC: 0.4946527296523447


### Conclusiones

Probando 10 configuraciones distintas del perceptrón multicapa y todas
dieron ROC-AUC ≈ 0.50. La razón no es que la red esté mal armada: es que este dataset es sintético y la etiqueta Tumor_Type se generó de forma prácticamente independiente del resto de las columnas. No hay patrón que aprender. Ningún modelo va a superar el azar con estos datos.